# Linear Regression Baseline Comparison

This notebook compares two future baseline models using the same Linear Regression setup:

- Actual boundary values -> price
- Forecast boundary values -> price

Both models use the same feature set: `System_Load`, `Wind_Power`, `Photovoltaic`, `Tie_Line`, `Hydropower`, and `Non-Marketized_Unit`.

In [ ]:
from pathlib import Path
import os


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        actual_path = candidate / "output/clean_datasets/price_prediction_dataset.csv"
        forecast_path = candidate / "output/clean_datasets/price_prediction_forecast_dataset.csv"
        if actual_path.exists() and forecast_path.exists():
            return candidate
    raise FileNotFoundError("Could not find both clean price-prediction datasets.")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ACTUAL_INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_dataset.csv"
FORECAST_INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_forecast_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "output/baseline_results/linear_comparison"

FEATURE_COLUMNS = [
    "System_Load",
    "Wind_Power",
    "Photovoltaic",
    "Tie_Line",
    "Hydropower",
    "Non-Marketized_Unit",
]
TARGET_COLUMN = "price"
MODEL_NAME = "Linear Regression"
RENEWABLE_FEATURES = "Wind_Power + Photovoltaic"

TRAIN_START = pd.Timestamp("2025-01-02 00:00:00")
TRAIN_END = pd.Timestamp("2025-10-31 23:45:00")
TEST_START = pd.Timestamp("2025-11-01 00:00:00")
TEST_END = pd.Timestamp("2025-12-31 23:45:00")

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Actual dataset: {ACTUAL_INPUT_PATH}")
print(f"Forecast dataset: {FORECAST_INPUT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## Load Datasets

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path, parse_dates=["time"])
    df = df.sort_values("time").reset_index(drop=True)

    required_columns = ["time", *FEATURE_COLUMNS, TARGET_COLUMN]
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise KeyError(f"Missing required columns in {path}: {missing_columns}")

    return df[required_columns].copy()


actual_df = load_dataset(ACTUAL_INPUT_PATH)
forecast_df = load_dataset(FORECAST_INPUT_PATH)

summary = pd.DataFrame(
    {
        "Actual": {
            "rows": len(actual_df),
            "start": actual_df["time"].min(),
            "end": actual_df["time"].max(),
        },
        "Forecast": {
            "rows": len(forecast_df),
            "start": forecast_df["time"].min(),
            "end": forecast_df["time"].max(),
        },
    }
).T

display(summary)
display(actual_df.head())
display(forecast_df.head())

## Train-Test Split

- Train: 2025-01-02 through 2025-10-31
- Test: 2025-11-01 through 2025-12-31

In [ ]:
def split_dataset(df):
    train = df[(df["time"] >= TRAIN_START) & (df["time"] <= TRAIN_END)].copy()
    test = df[(df["time"] >= TEST_START) & (df["time"] <= TEST_END)].copy()
    return train, test


actual_train, actual_test = split_dataset(actual_df)
forecast_train, forecast_test = split_dataset(forecast_df)

split_summary = pd.DataFrame(
    {
        "Actual train": {"rows": len(actual_train), "start": actual_train["time"].min(), "end": actual_train["time"].max()},
        "Actual test": {"rows": len(actual_test), "start": actual_test["time"].min(), "end": actual_test["time"].max()},
        "Forecast train": {"rows": len(forecast_train), "start": forecast_train["time"].min(), "end": forecast_train["time"].max()},
        "Forecast test": {"rows": len(forecast_test), "start": forecast_test["time"].min(), "end": forecast_test["time"].max()},
    }
).T

display(split_summary)

## Fit Linear Regression Baselines

In [ ]:
def make_linear_model():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    )


def metrics_row(data_type, split, y_true, y_pred):
    return {
        "data_type": data_type,
        "model": MODEL_NAME,
        "renewable_features": RENEWABLE_FEATURES,
        "split": split,
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def fit_evaluate(data_type, train_df, test_df):
    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET_COLUMN]
    X_test = test_df[FEATURE_COLUMNS]
    y_test = test_df[TARGET_COLUMN]

    model = make_linear_model()
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    metrics = pd.DataFrame(
        [
            metrics_row(data_type, "Train", y_train, train_pred),
            metrics_row(data_type, "Test", y_test, test_pred),
        ]
    )

    predictions = test_df[["time", TARGET_COLUMN]].copy()
    predictions["data_type"] = data_type
    predictions["predicted_price"] = test_pred
    predictions["residual"] = predictions[TARGET_COLUMN] - predictions["predicted_price"]

    coefficients = pd.DataFrame(
        {
            "data_type": data_type,
            "model": MODEL_NAME,
            "feature": FEATURE_COLUMNS,
            "coefficient_per_1_std": model.named_steps["model"].coef_,
        }
    )

    return model, metrics, predictions, coefficients


actual_model, actual_metrics, actual_predictions, actual_coefficients = fit_evaluate(
    "Actual",
    actual_train,
    actual_test,
)
forecast_model, forecast_metrics, forecast_predictions, forecast_coefficients = fit_evaluate(
    "Forecast",
    forecast_train,
    forecast_test,
)

metrics = pd.concat([actual_metrics, forecast_metrics], ignore_index=True)
test_predictions = pd.concat([actual_predictions, forecast_predictions], ignore_index=True)
coefficients = pd.concat([actual_coefficients, forecast_coefficients], ignore_index=True)

display(metrics)

coefficients_display = coefficients.assign(
    abs_coefficient=coefficients["coefficient_per_1_std"].abs()
).sort_values(["data_type", "abs_coefficient"], ascending=[True, False])
display(coefficients_display.drop(columns="abs_coefficient"))

## Test Prediction Plots

In [ ]:
def plot_first_week_predictions(predictions, data_type):
    plot_window = predictions[predictions["data_type"] == data_type].head(7 * 24 * 4)

    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.plot(plot_window["time"], plot_window[TARGET_COLUMN], label="Actual price", color="#111111", linewidth=1.8)
    ax.plot(plot_window["time"], plot_window["predicted_price"], label="Predicted price", color="#2166ac", alpha=0.9)
    ax.set_title(f"{data_type} Linear Regression Test Predictions: First 7 Days")
    ax.set_xlabel("Time")
    ax.set_ylabel("Standardized price")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig, ax


fig_actual_pred, ax = plot_first_week_predictions(test_predictions, "Actual")
display(fig_actual_pred)

fig_forecast_pred, ax = plot_first_week_predictions(test_predictions, "Forecast")
display(fig_forecast_pred)

## RMSE And R2 Bar Charts

In [ ]:
def plot_metric_comparison(metric, ylabel, title):
    split_order = ["Train", "Test"]
    data_types = ["Actual", "Forecast"]
    colors = ["#2166ac", "#1b9e77"]
    x = np.arange(len(split_order))
    width = 0.36

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for offset_idx, (data_type, color) in enumerate(zip(data_types, colors)):
        values = metrics[metrics["data_type"] == data_type].set_index("split").loc[split_order, metric]
        positions = x + (offset_idx - 0.5) * width
        bars = ax.bar(positions, values, width=width, label=data_type, color=color)
        ax.bar_label(bars, labels=[f"{value:.3f}" for value in values], padding=3, fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(split_order)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    return fig, ax


fig_rmse, ax = plot_metric_comparison(
    "RMSE",
    "RMSE",
    "Linear Regression RMSE: Actual vs Forecast Inputs",
)
display(fig_rmse)

fig_r2, ax = plot_metric_comparison(
    "R2",
    "R2",
    "Linear Regression R2: Actual vs Forecast Inputs",
)
display(fig_r2)

## Save Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics.to_csv(OUTPUT_DIR / "linear_baseline_metrics.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "linear_baseline_test_predictions.csv", index=False)
coefficients.to_csv(OUTPUT_DIR / "linear_baseline_coefficients.csv", index=False)

fig_actual_pred.savefig(OUTPUT_DIR / "actual_linear_test_predictions_first_week.png", bbox_inches="tight")
fig_forecast_pred.savefig(OUTPUT_DIR / "forecast_linear_test_predictions_first_week.png", bbox_inches="tight")
fig_rmse.savefig(OUTPUT_DIR / "linear_rmse_actual_vs_forecast.png", bbox_inches="tight")
fig_r2.savefig(OUTPUT_DIR / "linear_r2_actual_vs_forecast.png", bbox_inches="tight")

print(f"Saved outputs to: {OUTPUT_DIR}")